# Arbori decizionali vinuri

In [2]:
import numpy as np
import pandas as pd

In [3]:
col_names = ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']
data = pd.read_csv(r"C:\Users\colit\Desktop\Master 1\Metode Avansate de Analiza a Datelor\Proiect - Random Forest\Dataset\winequality-red-fixed-csv.csv", skiprows=1, header=None, names=col_names)
data.head(5)

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11,34,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25,67,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15,54,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17,60,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11,34,0.9978,3.51,0.56,9.4,5


In [4]:
class Node():
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, info_gain=None, value=None):
        '''constructor'''
        #pentru nodul de decizie
        self.feature_index = feature_index # pentru conditia nodului de decizie, indexul coloanei care sa va lua
        self.threshold = threshold # pentru valoarea conditiei 
        self.left = left # accesarea nodul copil din stanga
        self.right = right # accesarea nodul copil din dreapta 
        self.info_gain = info_gain

        #pentru nodul periferic
        self.value = value

In [5]:
class DecisionTreeClasifier():
    def __init__(self, min_samples_split=2,max_depth=2):
        '''constructor'''
        # se initializeaza radacina arborelui
        self.root = None

        # conditiile de oprire
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth

    def build_tree(self, dataset, curr_depth=0):
        '''functie recursiva de a construi arborele'''

        X,Y = dataset[:,:-1], dataset[:,-1]
        num_samples, num_features = np.shape(X)

        # despartem pana cand conditiile de oprire se indeplinesc
        if num_samples >= self.min_samples_split and curr_depth <= self.max_depth:

            # cautam cea mai buna conditie de separare
            best_split = self.get_best_split(dataset, num_samples, num_features)

            # verificam daca info_gain este pozitiv
            if best_split['info_gain'] > 0:

                # creem nodul din stanga recursiv
                left_subtree = self.build_tree(best_split['dataset_left'], curr_depth + 1)

                # calculam nodul din drepata recursiv
                right_subtree = self.build_tree(best_split['dataset_right'], curr_depth + 1)

                # returnam nodul de decizie
                return Node(best_split['feature_index'],best_split['threshold'], left_subtree, right_subtree, best_split['info_gain'])
        
        # calculam nodul periferic
        leaf_value = self.calculate_leaf_value(Y)
        
        # returnam nodul periferic
        return Node(value=leaf_value)
    
    def get_best_split(self, dataset, num_samples, num_features):
        '''functie de a gasi cea mai buna conditie de despartire'''

        # dictionar pentru a salva cea mai buna conditie de separare
        best_split = {}
        max_info_gain = -float("inf") # avem nevoie de valoare maxima negativa pentru a putea lua in calcul orice schimbare pozitiva

        # trecem prin toate caracteristicile (features -> coloanele)
        for feature_index in range(num_features):
            feature_values = dataset[:,feature_index]
            possible_threshold = np.unique(feature_values)
            # trecem prin toate valorile caracteristicilor (coloanelor -> features) prezente in datele noaste

            for threshold in possible_threshold:
                # luam conditia de separare curenta 
                dataset_left, dataset_right = self.split(dataset, feature_index, threshold)
                
                # verificam daca nodurile copil nu sunt nule
                if len(dataset_left) > 0 and len(dataset_right) > 0 :
                    y, left_y, right_y = dataset[:,-1], dataset_left[:,-1], dataset_right[:,-1]

                    #calculam info_gain
                    curr_info_gain = self.information_gain(y, left_y, right_y, 'gini')

                    #actualizam best_split, daca trebuie
                    if curr_info_gain > max_info_gain :
                        best_split['feature_index'] = feature_index
                        best_split['threshold'] = threshold
                        best_split['dataset_left'] = dataset_left
                        best_split['dataset_right'] = dataset_right
                        best_split['info_gain'] = curr_info_gain
                        max_info_gain = curr_info_gain
            
        # returnam cea mai buna conditie de separare (best_split)
        return best_split
    
    # functia de impartire a datelor conform conditiei din nod
    def split(self, dataset, feature_index, threshold):
        dataset_left = np.array([row for row in dataset if row[feature_index] <= threshold])
        dataset_right = np.array([row for row in dataset if row[feature_index] > threshold])

        return dataset_left,dataset_right

    #functia information gain
    def information_gain(self, parent, l_child, r_child, mode='entropy'):
        weight_l = len(l_child)/len(parent)
        weight_r = len(r_child)/len(parent)
        if mode == 'gini':
            gain = self.gini_index(parent) - (weight_l * self.gini_index(l_child) + weight_r * self.gini_index(r_child))
        else:
            gain = self.entropy(parent) - (weight_l * self.entropy(l_child) + weight_r * self.entropy(r_child))
        
        return gain

    # functia entropy
    def entropy(self, y):
        class_labels = np.unique(y)
        entropy = 0
        for cls in class_labels:
            p_cls = len(y[y == cls]) / len(y)
            entropy += p_cls * np.log2(p_cls)
        
        return entropy
    
    # functia gini_index
    def gini_index(self, y):
        class_labels = np.unique(y)
        gini = 0
        for cls in class_labels:
            p_cls = len(y[y == cls]) / len(y)
            gini += p_cls**2
        return 1 - gini
    
    # functia de caluclare a valorii nodului terminal
    def calculate_leaf_value(self, Y):
        Y = list(Y)
        return max(Y, key=Y.count)
    
    def print_tree(self, tree=None, indent=" "):
        if not tree:
            tree = self.root
        
        if tree.value is not None:
            print(tree.value)
        
        else:
            print("X_"+str(tree.feature_index), "<=", tree.threshold, "?", tree.info_gain)
            print("%sleft:" % (indent), end="")
            self.print_tree(tree.left, indent + indent)
            print("%sright:" % (indent), end="")
            self.print_tree(tree.right, indent + indent)

    # functia de antrenare
    def fit(self, X, Y):
        dataset = np.concatenate((X, Y), axis=1)
        self.root = self.build_tree(dataset)
    
    # functia de predictie
    def predict(self, X):
        predictions = [self.make_prediction(x, self.root) for x in X]
        return predictions
    
    def make_prediction(self, x, tree):
        if tree.value != None: return tree.value
        feature_val = x[tree.feature_index]
        if feature_val <= tree.threshold:
            return self.make_prediction(x, tree.left)
        else:
            return self.make_prediction(x, tree.right)

In [6]:
X = data.iloc[:, :-1].values
Y = data.iloc[:, -1].values.reshape(-1,1)
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, random_state=41)

In [7]:
classifier = DecisionTreeClasifier(min_samples_split=11,max_depth=11)
classifier.fit(X_train, Y_train)
classifier.print_tree()

X_10 <= 10.2 ? 0.0675689446039579
 left:X_6 <= 98.0 ? 0.01979183427826703
  left:X_9 <= 0.57 ? 0.019946445538223778
    left:X_10 <= 9.05 ? 0.019121392030038986
        left:X_0 <= 7.4 ? 0.25560802833530105
                left:4.0
                right:6.0
        right:X_10 <= 9.7 ? 0.020095164953136013
                left:X_8 <= 3.53 ? 0.018396133780749113
                                left:X_3 <= 4.3 ? 0.013842097935186037
                                                                left:X_1 <= 0.23 ? 0.014141974973855276
                                                                                                                                left:6.0
                                                                                                                                right:X_6 <= 51.0 ? 0.011437954233718906
                                                                                                                                                            

In [8]:
Y_pred = classifier.predict(X_test)
from sklearn.metrics import accuracy_score
accuracy_score(Y_test, Y_pred)

0.6

In [9]:
print(X_train[12], Y_train[12])

[ 6.4     0.4     0.23    1.6     0.066   5.     12.      0.9958  3.34
  0.56    9.2   ] [5]


In [10]:
wine8 = [7.9, 0.35, 0.46, 3.6, 0.078, 15, 37, 0.9973, 3.35, 0.86, 12.8]
entity = [8.000e+00, 7.450e-01, 5.600e-01, 2.000e+00, 1.180e-01, 3.000e+01, 1.340e+02, 9.968e-01, 3.240e+00, 6.600e-01, 9.400e+00]
print(classifier.make_prediction(wine8, classifier.root))

6.0


# Ce as putea face

1. Vad rezultatele si cu random forest.
2. Sa ma folosesc de vreun algoritm de reducere a datelor.